In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Copy the 'data' folder from Google Drive to the current Colab environment
!cp -r "/content/drive/My Drive/data" "/content/"

# List the contents of the copied 'data' folder to verify
print("Contents of the copied 'data' folder:")
!ls /content/data

Contents of the copied 'data' folder:
final_submission  test	train


In [4]:
import pandas as pd
import numpy as np
import json
import warnings
!pip install catboost

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.2 MB/s eta 0:00:00


In [7]:
train_flag = pd.read_csv("/content/data/train/train_flag.csv")
test_flag = pd.read_csv("/content/data/test/test_flag.csv")

sample_submission = pd.read_csv("/content/data/final_submission/sample_submission.csv")

print(train_flag.shape)
print(test_flag.shape)

(261383, 3)
(46127, 2)


In [8]:
with open("/content/data/train/accounts_data_train.json") as f:
    acc_train = json.load(f)

with open("/content/data/test/accounts_data_test.json") as f:
    acc_test = json.load(f)

with open("/content/data/train/enquiry_data_train.json") as f:
    enq_train = json.load(f)

with open("/content/data/test/enquiry_data_test.json") as f:
    enq_test = json.load(f)

Flatten Accounts Train

In [9]:
accounts_rows = []

for customer in acc_train:
    for loan in customer:
        accounts_rows.append(loan)

accounts_df = pd.DataFrame(accounts_rows)

print(accounts_df.shape)
accounts_df.head()

(1245310, 7)


,credit_type,loan_amount,amount_overdue,open_date,closed_date,payment_hist_string,uid
0,Consumer credit,272745.000,0.0,2018-09-22,2020-02-22,0000000000000000000000100000000000000000000000...,AAA09044550
1,Consumer credit,4500.000,0.0,2018-03-08,2019-07-25,000000000000000014044000000000000000000000000000,AAA09044550
2,Credit card,80996.445,0.0,2020-06-29,NaN,000000000000000000,AAA10545297
3,Consumer credit,43771.500,0.0,2020-06-09,2020-09-09,000000000,AAA14112888
4,Credit card,10480.500,0.0,2014-09-10,NaN,0000000000000000000000000000000000000000000000...,AAA20326915


Flatten Accounts Test

In [10]:
accounts_rows_test = []

for customer in acc_test:
    for loan in customer:
        accounts_rows_test.append(loan)

accounts_test_df = pd.DataFrame(accounts_rows_test)

print(accounts_test_df.shape)
accounts_test_df.head()

(220013, 7)


,credit_type,loan_amount,amount_overdue,open_date,closed_date,payment_hist_string,uid
0,Consumer credit,31630.50,0.0,2014-03-30,2014-11-29,000000000000000000000000,AAA14437029
1,Consumer credit,14613.39,0.0,2014-06-01,2014-11-03,000000000000000,AAA14437029
2,Credit card,54000.00,0.0,2015-12-13,2019-09-21,0000000000000000000000000000000000000000000000...,AAA14437029
3,Consumer credit,27076.50,0.0,2015-11-11,2016-11-24,000000000000000000000000000000000000,AAA14437029
4,Credit card,225000.00,0.0,2017-07-15,2019-11-14,0000000000000000000000000000000000000000000000...,AAA14437029


Flatten Enquiry Train

In [11]:
enquiry_rows = []

for customer in enq_train:
    for enquiry in customer:
        enquiry_rows.append(enquiry)

enquiry_df = pd.DataFrame(enquiry_rows)

print(enquiry_df.shape)
enquiry_df.head()

(1909926, 4)


,enquiry_type,enquiry_amt,enquiry_date,uid
0,Interbank credit,168839,2020-11-08,AAA08065248
1,Mobile operator loan,268392,2020-09-20,AAA08065248
2,Mobile operator loan,36082,2020-06-19,AAA08065248
3,Interbank credit,180467,2019-10-22,AAA08065248
4,Cash loan (non-earmarked),227459,2020-05-24,AAA08065248


Flatten Enquiry Test

In [12]:
enquiry_rows_test = []

for customer in enq_test:
    for enquiry in customer:
        enquiry_rows_test.append(enquiry)

enquiry_test_df = pd.DataFrame(enquiry_rows_test)

print(enquiry_test_df.shape)
enquiry_test_df.head()

(337662, 4)


,enquiry_type,enquiry_amt,enquiry_date,uid
0,Car loan,143000,2020-12-13,AAA02107680
1,Real estate loan,174000,2020-12-01,AAA14437029
2,Loan for working capital replenishment,65000,2019-07-01,AAA14437029
3,Loan for working capital replenishment,118000,2020-08-05,AAA14437029
4,Car loan,12000,2020-02-28,AAA14437029


Payment History Parser

In [13]:
def parse_payment_hist(hist):

    if pd.isna(hist):
        return []

    hist = str(hist)

    return [
        int(hist[i:i+3])
        for i in range(0, len(hist), 3)
    ]

Advanced Payment Features

In [14]:
def payment_features(hist):

    vals = parse_payment_hist(hist)

    if len(vals) == 0:

        return pd.Series({

            "max_dpd": 0,
            "mean_dpd": 0,
            "recent_dpd": 0,
            "std_dpd": 0,

            "late_1": 0,
            "late_30": 0,
            "late_60": 0,
            "late_90": 0,

            "months_history": 0,

            "num_late": 0,

            "last_3m_late": 0,
            "last_6m_late": 0,

            "recent_mean": 0,
            "older_mean": 0,

            "dpd_trend": 0,

            "max_consecutive_late": 0,

            "late_ratio": 0,

            "worst_recent_6m": 0,

            "recent_vs_old_ratio": 0
        })

    vals = np.array(vals)

    recent_3 = vals[-3:] if len(vals) >= 3 else vals

    recent_6 = vals[-6:] if len(vals) >= 6 else vals

    older = vals[:-6] if len(vals) > 6 else vals

    recent_mean = recent_6.mean()

    older_mean = older.mean()

    trend = recent_mean - older_mean

    max_streak = 0
    streak = 0

    for v in vals:

        if v > 0:
            streak += 1
            max_streak = max(max_streak, streak)

        else:
            streak = 0

    late_ratio = np.mean(vals > 0)

    return pd.Series({

        "max_dpd": vals.max(),

        "mean_dpd": vals.mean(),

        "recent_dpd": vals[-1],

        "std_dpd": vals.std(),

        "late_1": np.sum(vals > 0),

        "late_30": np.sum(vals >= 30),

        "late_60": np.sum(vals >= 60),

        "late_90": np.sum(vals >= 90),

        "months_history": len(vals),

        "num_late": np.sum(vals > 0),

        "last_3m_late": np.sum(recent_3 > 0),

        "last_6m_late": np.sum(recent_6 > 0),

        "recent_mean": recent_mean,

        "older_mean": older_mean,

        "dpd_trend": trend,

        "max_consecutive_late": max_streak,

        "late_ratio": late_ratio,

        "worst_recent_6m": recent_6.max(),

        "recent_vs_old_ratio":
            recent_mean / (older_mean + 1)
    })

Apply Payment Features

In [15]:
payment_feats = accounts_df["payment_hist_string"].apply(payment_features)

accounts_df = pd.concat(
    [accounts_df, payment_feats],
    axis=1
)

payment_feats_test = accounts_test_df["payment_hist_string"].apply(payment_features)

accounts_test_df = pd.concat(
    [accounts_test_df, payment_feats_test],
    axis=1
)

Date Features

In [16]:
for df in [accounts_df, accounts_test_df]:

    df["open_date"] = pd.to_datetime(df["open_date"])

    df["closed_date"] = pd.to_datetime(df["closed_date"])

    df["loan_duration"] = (
        df["closed_date"] - df["open_date"]
    ).dt.days

    df["loan_duration"] = df["loan_duration"].fillna(0)

    df["is_active"] = df["closed_date"].isna().astype(int)

    df["overdue_ratio"] = (
        df["amount_overdue"] /
        (df["loan_amount"] + 1)
    )

    df["active_loan_amount"] = (
        df["loan_amount"] *
        df["is_active"]
    )

    df["active_overdue"] = (
        df["amount_overdue"] *
        df["is_active"]
    )

Frequency Encoding

In [17]:
credit_freq = accounts_df["credit_type"].value_counts()

accounts_df["credit_type_freq"] = (
    accounts_df["credit_type"]
    .map(credit_freq)
)

accounts_test_df["credit_type_freq"] = (
    accounts_test_df["credit_type"]
    .map(credit_freq)
)

enquiry_freq = enquiry_df["enquiry_type"].value_counts()

enquiry_df["enquiry_type_freq"] = (
    enquiry_df["enquiry_type"]
    .map(enquiry_freq)
)

enquiry_test_df["enquiry_type_freq"] = (
    enquiry_test_df["enquiry_type"]
    .map(enquiry_freq)
)

Aggregate Accounts Train

In [18]:
agg_accounts = accounts_df.groupby("uid").agg({

    "loan_amount": ["sum", "mean", "max", "std", "count"],

    "amount_overdue": ["sum", "mean", "max"],

    "overdue_ratio": ["mean", "max"],

    "loan_duration": ["mean", "max"],

    "active_loan_amount": ["sum", "mean"],

    "active_overdue": ["sum", "mean"],

    "max_dpd": ["max", "mean"],

    "mean_dpd": ["mean", "max"],

    "recent_dpd": ["max", "mean"],

    "std_dpd": ["mean"],

    "late_1": ["sum"],

    "late_30": ["sum"],

    "late_60": ["sum"],

    "late_90": ["sum"],

    "num_late": ["sum"],

    "last_3m_late": ["sum"],

    "last_6m_late": ["sum"],

    "recent_mean": ["mean"],

    "older_mean": ["mean"],

    "dpd_trend": ["mean"],

    "max_consecutive_late": ["max"],

    "late_ratio": ["mean"],

    "worst_recent_6m": ["max"],

    "recent_vs_old_ratio": ["mean"],

    "months_history": ["sum", "mean"],

    "credit_type_freq": ["mean"],

    "is_active": ["sum"]

})

agg_accounts.columns = [
    "_".join(col).strip()
    for col in agg_accounts.columns.values
]

agg_accounts.reset_index(inplace=True)

Aggregate Accounts Test

In [19]:
agg_accounts_test = accounts_test_df.groupby("uid").agg({

    "loan_amount": ["sum", "mean", "max", "std", "count"],

    "amount_overdue": ["sum", "mean", "max"],

    "overdue_ratio": ["mean", "max"],

    "loan_duration": ["mean", "max"],

    "active_loan_amount": ["sum", "mean"],

    "active_overdue": ["sum", "mean"],

    "max_dpd": ["max", "mean"],

    "mean_dpd": ["mean", "max"],

    "recent_dpd": ["max", "mean"],

    "std_dpd": ["mean"],

    "late_1": ["sum"],

    "late_30": ["sum"],

    "late_60": ["sum"],

    "late_90": ["sum"],

    "num_late": ["sum"],

    "last_3m_late": ["sum"],

    "last_6m_late": ["sum"],

    "recent_mean": ["mean"],

    "older_mean": ["mean"],

    "dpd_trend": ["mean"],

    "max_consecutive_late": ["max"],

    "late_ratio": ["mean"],

    "worst_recent_6m": ["max"],

    "recent_vs_old_ratio": ["mean"],

    "months_history": ["sum", "mean"],

    "credit_type_freq": ["mean"],

    "is_active": ["sum"]

})

agg_accounts_test.columns = [
    "_".join(col).strip()
    for col in agg_accounts_test.columns.values
]

agg_accounts_test.reset_index(inplace=True)

Enquiry Features

In [20]:
reference_date = pd.Timestamp("2021-01-01")

for df in [enquiry_df, enquiry_test_df]:

    df["enquiry_date"] = pd.to_datetime(df["enquiry_date"])

    df["days_since_enquiry"] = (
        reference_date - df["enquiry_date"]
    ).dt.days

    df["recent_30d"] = (
        df["days_since_enquiry"] <= 30
    ).astype(int)

    df["recent_90d"] = (
        df["days_since_enquiry"] <= 90
    ).astype(int)

Aggregate Enquiry Train

In [21]:
agg_enquiry = enquiry_df.groupby("uid").agg({

    "enquiry_amt": ["count", "sum", "mean", "max", "std"],

    "days_since_enquiry": ["min", "mean"],

    "recent_30d": ["sum"],

    "recent_90d": ["sum"],

    "enquiry_type_freq": ["mean"]

})

agg_enquiry.columns = [
    "_".join(col).strip()
    for col in agg_enquiry.columns.values
]

agg_enquiry.reset_index(inplace=True)

Aggregate Enquiry Test

In [22]:
agg_enquiry_test = enquiry_test_df.groupby("uid").agg({

    "enquiry_amt": ["count", "sum", "mean", "max", "std"],

    "days_since_enquiry": ["min", "mean"],

    "recent_30d": ["sum"],

    "recent_90d": ["sum"],

    "enquiry_type_freq": ["mean"]

})

agg_enquiry_test.columns = [
    "_".join(col).strip()
    for col in agg_enquiry_test.columns.values
]

agg_enquiry_test.reset_index(inplace=True)

Credit Diversity

In [23]:
credit_div_train = accounts_df.groupby("uid")["credit_type"] \
    .nunique() \
    .reset_index(name="num_credit_types")

credit_div_test = accounts_test_df.groupby("uid")["credit_type"] \
    .nunique() \
    .reset_index(name="num_credit_types")

Merge Train

In [24]:
train = train_flag.merge(
    agg_accounts,
    on="uid",
    how="left"
)

train = train.merge(
    agg_enquiry,
    on="uid",
    how="left"
)

train = train.merge(
    credit_div_train,
    on="uid",
    how="left"
)

Merge Test

In [25]:
test = test_flag.merge(
    agg_accounts_test,
    on="uid",
    how="left"
)

test = test.merge(
    agg_enquiry_test,
    on="uid",
    how="left"
)

test = test.merge(
    credit_div_test,
    on="uid",
    how="left"
)

Fill Missing

In [26]:
train.fillna(0, inplace=True)
test.fillna(0, inplace=True)

Ratio Features

In [27]:
for df in [train, test]:

    df["overdue_to_loan"] = (
        df["amount_overdue_sum"] /
        (df["loan_amount_sum"] + 1)
    )

    df["late_per_month"] = (
        df["late_1_sum"] /
        (df["months_history_sum"] + 1)
    )

    df["recent_to_max_dpd"] = (
        df["recent_dpd_max"] /
        (df["max_dpd_max"] + 1)
    )

    df["active_ratio"] = (
        df["active_loan_amount_sum"] /
        (df["loan_amount_sum"] + 1)
    )

Prepare Model Data

In [28]:
X = train.drop(["uid", "TARGET"], axis=1)

y = train["TARGET"]

X_test = test.drop(["uid"], axis=1)

cat_features = ["NAME_CONTRACT_TYPE"]

Log Transform

In [29]:
for col in X.columns:

    if "amount" in col or "loan" in col:

        X[col] = np.log1p(X[col])

        X_test[col] = np.log1p(X_test[col])

Cross Validation

In [30]:
folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_cat = np.zeros(len(X))
preds_cat = np.zeros(len(X_test))

oof_lgb = np.zeros(len(X))
preds_lgb = np.zeros(len(X_test))

CatBoost Training

In [31]:
for fold, (train_idx, valid_idx) in enumerate(folds.split(X, y)):

    print("=" * 50)
    print(f"CATBOOST FOLD {fold+1}")
    print("=" * 50)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(

        iterations=4000,

        learning_rate=0.01,

        depth=8,

        l2_leaf_reg=5,

        loss_function="Logloss",

        eval_metric="AUC",

        random_seed=42,

        verbose=200
    )

    model.fit(

        X_train,
        y_train,

        cat_features=cat_features,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=300
    )

    oof_cat[valid_idx] = model.predict_proba(X_valid)[:, 1]

    preds_cat += model.predict_proba(X_test)[:, 1] / 5

CATBOOST FOLD 1
0:	test: 0.5068315	best: 0.5068315 (0)	total: 186ms	remaining: 12m 23s
200:	test: 0.6709715	best: 0.6709715 (200)	total: 41.7s	remaining: 13m 8s
400:	test: 0.6783463	best: 0.6783463 (400)	total: 1m 16s	remaining: 11m 23s
600:	test: 0.6804128	best: 0.6804128 (600)	total: 1m 49s	remaining: 10m 17s
800:	test: 0.6819191	best: 0.6819191 (800)	total: 2m 25s	remaining: 9m 41s
1000:	test: 0.6825719	best: 0.6825719 (1000)	total: 2m 59s	remaining: 8m 56s
1200:	test: 0.6829142	best: 0.6829591 (1166)	total: 3m 32s	remaining: 8m 14s
1400:	test: 0.6833472	best: 0.6833472 (1400)	total: 4m 5s	remaining: 7m 36s
1600:	test: 0.6833447	best: 0.6833857 (1412)	total: 4m 41s	remaining: 7m 2s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.6833856534
bestIteration = 1412

Shrink model to first 1413 iterations.
CATBOOST FOLD 2
0:	test: 0.5072393	best: 0.5072393 (0)	total: 137ms	remaining: 9m 8s
200:	test: 0.6598376	best: 0.6598376 (200)	total: 33.3s	remaining: 10m 28s
400:	

CatBoost AUC

In [32]:
cat_auc = roc_auc_score(y, oof_cat)

print("CATBOOST AUC =", cat_auc)

CATBOOST AUC = 0.6791786859709164


LightGBM Training

In [34]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X["NAME_CONTRACT_TYPE"] = le.fit_transform(
    X["NAME_CONTRACT_TYPE"]
)

X_test["NAME_CONTRACT_TYPE"] = le.transform(
    X_test["NAME_CONTRACT_TYPE"]
)

In [37]:
for fold, (train_idx, valid_idx) in enumerate(folds.split(X, y)):

    print("=" * 50)
    print(f"LGBM FOLD {fold+1}")
    print("=" * 50)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = LGBMClassifier(

        n_estimators=3000,

        learning_rate=0.02,

        objective="binary",

        boosting_type="gbdt",

        num_leaves=31,

        max_depth=-1,

        min_child_samples=50,

        subsample=0.8,

        colsample_bytree=0.8,

        reg_alpha=1,

        reg_lambda=1,

        random_state=42,

        n_jobs=-1
    )

    model.fit(

        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)]
    )

    oof_lgb[valid_idx] = model.predict_proba(X_valid)[:, 1]

    preds_lgb += model.predict_proba(X_test)[:, 1] / 5

LGBM FOLD 1
[LightGBM] [Info] Number of positive: 16846, number of negative: 192260
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.506760 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11225
[LightGBM] [Info] Number of data points in the train set: 209106, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080562 -> initscore=-2.434735
[LightGBM] [Info] Start training from score -2.434735
LGBM FOLD 2
[LightGBM] [Info] Number of positive: 16845, number of negative: 192261
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.082873 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11234
[LightGBM] [Info] Number of data points in the train set: 209106, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.08

LightGBM AUC

In [38]:
lgb_auc = roc_auc_score(y, oof_lgb)

print("LIGHTGBM AUC =", lgb_auc)

LIGHTGBM AUC = 0.6696560203330306


Blend Predictions

In [39]:
oof_blend = (
    0.5 * oof_cat +
    0.5 * oof_lgb
)

preds_blend = (
    0.5 * preds_cat +
    0.5 * preds_lgb
)

blend_auc = roc_auc_score(y, oof_blend)

print("BLENDED AUC =", blend_auc)

BLENDED AUC = 0.6770258111521168


Feature Importance

In [40]:
feature_imp = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_
})

feature_imp = feature_imp.sort_values(
    by="Importance",
    ascending=False
)

feature_imp.head(30)

,Feature,Importance
48,days_since_enquiry_mean,5965
51,enquiry_type_freq_mean,4861
44,enquiry_amt_mean,4692
47,days_since_enquiry_min,4549
39,months_history_mean,4261
43,enquiry_amt_sum,3865
45,enquiry_amt_max,3859
46,enquiry_amt_std,3776
38,months_history_sum,3692
12,loan_duration_max,3629


Submission

In [41]:
submission = sample_submission.copy()

submission["TARGET"] = preds_blend

submission.to_csv(
    "final_submission_tejasv_gupta.csv",
    index=False
)

print("Submission Saved")

Submission Saved
